In [1]:
import arxiv, os, urllib.request

os.makedirs("../data/papers", exist_ok=True)

client = arxiv.Client()
search = arxiv.Search(
    query="retrieval augmented generation for question answering",
    max_results=10,
    sort_by=arxiv.SortCriterion.Relevance
)

downloaded = []
for r in client.results(search):
    url = r.pdf_url
    filename = url.split("/")[-1] + ".pdf"
    filepath = os.path.join("../data/papers", filename)
    if not os.path.exists(filepath):
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Downloaded: {filename}")
        except Exception as e:
            print(f"Failed: {filename} — {e}")
            continue
    else:
        print(f"Already have: {filename}")
    downloaded.append({"title": r.title, "filepath": filepath, "url": url})

print(f"\nTotal papers: {len(downloaded)}")

Downloaded: 2507.23334v2.pdf
Downloaded: 2404.14464v1.pdf
Downloaded: 1912.02145v1.pdf
Downloaded: 2108.12898v1.pdf
Already have: 2506.06962v3.pdf
Downloaded: 2601.07528v2.pdf
Downloaded: 2504.13684v1.pdf
Downloaded: 2510.25621v1.pdf
Downloaded: 2504.17204v1.pdf
Downloaded: 1811.08772v1.pdf

Total papers: 10


In [2]:
import pymupdf, os, pickle

papers_dir = "../data/papers"
all_chunks = []

for fname in sorted(os.listdir(papers_dir)):
    if not fname.endswith(".pdf"):
        continue
    path = os.path.join(papers_dir, fname)
    try:
        doc = pymupdf.open(path)
        text = "".join(page.get_text() for page in doc)
    except Exception as e:
        print(f"Skipped {fname}: {e}")
        continue

    # chunk
    size, overlap = 500, 100
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+size])
        start += size - overlap

    # tag each chunk with its source paper
    for c in chunks:
        all_chunks.append({"text": c, "source": fname})

    print(f"{fname}: {len(chunks)} chunks")

print(f"\nTotal chunks across all papers: {len(all_chunks)}")

1811.08772v1.pdf: 145 chunks
1912.02145v1.pdf: 80 chunks
2108.12898v1.pdf: 73 chunks
2404.14464v1.pdf: 143 chunks
2504.13684v1.pdf: 67 chunks
2504.17204v1.pdf: 46 chunks
2506.06962v3.pdf: 172 chunks
2507.23334v2.pdf: 93 chunks
2510.25621v1.pdf: 325 chunks
2601.07528v2.pdf: 212 chunks

Total chunks across all papers: 1356


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings, dtype="float32"))
print("Index size:", index.ntotal)

/home/rgukt/Desktop/research-assistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|█████████████████████████████████| 43/43 [02:59<00:00,  4.18s/it]

Embeddings shape: (1356, 384)
Index size: 1356


In [4]:
def search_corpus(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb, dtype="float32"), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "text": all_chunks[idx]["text"],
            "source": all_chunks[idx]["source"]
        })
    return results

results = search_corpus("how does retrieval augmented generation improve question answering", k=5)

for i, r in enumerate(results, 1):
    print(f"--- {i}. score={r['score']:.3f} | {r['source']} ---")
    print(r["text"][:200])
    print()

--- 1. score=0.688 | 2510.25621v1.pdf ---
-
Augmented Generation.
2.2
The Evolution of
Retrieval-Augmented Generation
(RAG)
RAG (Lewis et al., 2020) was a seminal work
that formally combined a pre-trained retriever
with a sequence-to-sequence

--- 2. score=0.680 | 2507.23334v2.pdf ---
la, “Retrieval-
augmented generation for knowledge-intensive nlp
tasks,” 2021. [Online]. Available: https://arxiv.org/abs/
2005.11401
[5] Z. Zhao, E. Monti, J. Lehmann, and H. Assem, “En-
hancing cont

--- 3. score=0.675 | 2404.14464v1.pdf ---
trieval of in-
formation encoded in their weights (Maynez et al.,
2020). In response to this issue, arguments for
LLMs with knowledge from external data sources
have emerged as a promising approach, a

--- 4. score=0.653 | 2601.07528v2.pdf ---
g the
reliability of the annotation guidelines.
F
Retrieval Comparison
Our main experiments focus on end-to-end answer
faithfulness under grounded generation, rather than
isolating the retrieval engin

--- 5. score=0.650 | 250

In [5]:
import pickle
import faiss

with open("../data/all_chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

faiss.write_index(index, "../data/all_index.faiss")

print("  all_chunks.pkl:", len(all_chunks), "chunks")
print("  all_index.faiss:", index.ntotal, "vectors")


  all_chunks.pkl: 1356 chunks
  all_index.faiss: 1356 vectors
